[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook

# How origin ASes announce IPv4 prefixes in BGP

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
%pip install pybgpkit-parser pelicanfs

import gzip
import urllib.request
import bz2
import io
import statistics
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

import ipaddress

import matplotlib.pyplot as plt
import pybgpkit_parser as bgpkit
from pelicanfs.core import OSDFFileSystem


@contextmanager
def open_safe(filename, encoding="utf-8"):
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f

---

## Task 1: CCDF of Origin AS IPv4 Prefix and Address Counts

In this task you will:

1. Fetch a BGP routing table (RIB) snapshot from a RouteViews collector via OSDF.
2. Build a `prefix → set of origin ASes` mapping and identify MOAS prefixes (announced by more than one AS).
3. For single-origin IPv4 prefixes, compute each AS's **prefix count** and **address count**.
   The address count uses a longest-prefix-match stack to avoid double-counting nested prefixes.
4. Plot a **Complementary Cumulative Distribution Function (CCDF)** with two curves:
   - by prefix count (bottom x-axis)
   - by address count (top x-axis)

Both axes are log-scaled. Each point `(x, y)` means *y* ASes have at least *x* prefixes (or addresses).

In [ ]:
OSDF_BASE = "https://osdf-director.osg-htc.org"

collectors = ["route-views3"]


def prefix2as_for_collector(collector):
    osdf = OSDFFileSystem()
    path = f"/routeviews/{collector}/bgpdata/2026.03/RIBS"
    objects = sorted(osdf.ls(path), key=lambda x: x["name"])
    url = OSDF_BASE + objects[0]["name"]
    print(f"  reading {url}")

    prefix_origins = defaultdict(set)
    i = 0
    for elem in bgpkit.Parser(url=url):
        # YOUR CODE HERE
        # Add elem.origin_asns to prefix_origins[elem.prefix].
        # Increment i. Every 1,000,000 elements print progress.
        # Stop processing after 1,000,000 elements.
        pass
    return prefix_origins


collector = collectors[0]
print(f"processing {collector}...")
prefix_origins = prefix2as_for_collector(collector)

In [ ]:
moas_count = 0
asn_prefix_count = defaultdict(int)
asn_to_nets = defaultdict(list)
ipv4_single_origin = []  # (net, asn) for IPv4 single-origin prefixes
for prefix, origins in prefix_origins.items():
    net = ipaddress.ip_network(prefix, strict=False)
    if net.version != 4:
        continue
    # YOUR CODE HERE
    # If len(origins) > 1, this is a MOAS prefix: increment moas_count.
    # Otherwise extract the single origin: asn = next(iter(origins))
    #   - increment asn_prefix_count[asn]
    #   - append net to asn_to_nets[asn]
    #   - append (net, asn) to ipv4_single_origin

# Compute per-AS IPv4 address counts via longest-prefix-match (no double counting).
# Sort by (start, prefixlen) so less-specific containers appear before their children.
records = sorted(
    ((int(net.network_address), net.prefixlen, net.num_addresses, int(net.network_address) + net.num_addresses - 1, asn)
     for net, asn in ipv4_single_origin),
    key=lambda r: (r[0], r[1])
)
direct = [r[2] for r in records]  # direct[i] = addresses attributed to records[i]
stack = []  # indices of open containers
for i, (start, _, size, _, asn) in enumerate(records):
    # YOUR CODE HERE
    # Remove entries from the top of stack while the container they represent
    # has already ended: records[stack[-1]][3] < start.
    # If stack is non-empty after that, subtract size from direct[stack[-1]]
    # so the container's address count doesn't double-count this child prefix.
    # Append i to stack so future children can find this prefix as their container.
    pass
asn_address_count = defaultdict(int)
for i, (_, _, _, _, asn) in enumerate(records):
    asn_address_count[asn] += direct[i]

total_prefixes = len(prefix_origins)
single_count = total_prefixes - moas_count
single_pct = 100.0 * single_count / total_prefixes
moas_pct = 100.0 * moas_count / total_prefixes
counts_list = list(asn_prefix_count.values())
avg = sum(counts_list) / len(counts_list)
median = statistics.median(counts_list)

print(f"total prefixes:")
print(f"   total :          {total_prefixes}")
print(f"   moas:            {moas_count} ({moas_pct:.1f}%)")
print(f"   single asn:      {single_count} ({single_pct:.1f}%)")
print(f"origin (number of prefixes)")
print(f"   minimum:         {min(counts_list)}")
print(f"   maximum:         {max(counts_list)}")
print(f"   average per ASN: {avg:.1f}")
print(f"   median per ASN:  {int(median)}")

total_addr = sum(asn_address_count.values())
print(f"total IPv4 addresses attributed: {total_addr}")

In [ ]:
count_dist = Counter(counts_list)
# YOUR CODE HERE
# Compute CCDF arrays for prefix counts.
# Using count_dist and asn_prefix_count:
#   x_prefix: sorted list of unique prefix count values
#   total_asns: total number of origin ASes
#   y_asns: list where y_asns[i] = number of ASes with at least x_prefix[i] prefixes.
#     Hint: start remaining = total_asns, subtract count_dist[xi] at each step.
pass

# YOUR CODE HERE
# Compute CCDF arrays for address counts.
# Using asn_address_count (ASN -> address count, built in the cell above):
#   addr_counts_list: address counts from asn_address_count.values(), excluding zeros
#   addr_count_dist: Counter of addr_counts_list
#   x_addr: sorted list of unique address count values
#   y_addr: list where y_addr[i] = number of ASes with at least x_addr[i] addresses
pass

fig, ax1 = plt.subplots()
ax1.plot(x_prefix, y_asns, marker=".", markersize=5, color="C0", label="by prefix count (bottom axis)")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Number of IPv4 prefixes (bottom axis)")
ax1.set_ylabel("Number of ASNs")

ax2 = ax1.twiny()
ax2.plot(x_addr, y_addr, marker=".", markersize=5, color="C1", label="by address count")
ax2.set_xscale("log")
ax2.set_xlabel("Number of IPv4 addresses")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.set_title(f"CCDF: IPv4 prefix and address count per AS ({collector})")
ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

### Question 1

What does the shape of the CCDF reveal about how IPv4 prefixes are distributed among origin ASes?

YOUR ANSWER HERE

### Question 2

What percentage of IPv4 prefixes are MOAS (announced by more than one origin AS)? What does MOAS represent, and why does it matter for routing security?

YOUR ANSWER HERE

### Question 3

Why do the prefix-count curve and the address-count curve diverge on the plot? What does this tell you about how address space is allocated relative to prefix announcements?

YOUR ANSWER HERE

---

## Task 2: CCDF of Customer Cone IPv4 Prefix and Address Counts

In this task you will:

1. Load the CAIDA PPDC customer-cone file, which maps each root AS to the set of ASes in its customer cone.
2. Compute each root AS's **cone prefix count** and **cone address count** by aggregating counts across every cone member (reusing `asn_prefix_count` and `asn_to_nets` from Task 1).
3. Plot a **CCDF** with two curves on a log-log scale:
   - by cone prefix count (bottom x-axis)
   - by cone address count (top x-axis)

A prefix is counted toward a customer cone once per cone; the same prefix may appear in multiple cones.

In [ ]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)
if not AS_CONE_PATH.exists(): 
    print (f"Unable to save file {AS_CONE_PATH}")
    exit() 

print("loading customer cone data...")
asn_to_cone = {}
with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        # YOUR CODE HERE
        # Split the line into parts. The first token is the root ASN;
        # the remaining tokens are all ASNs in the cone (including the root).
        # Store as: asn_to_cone[int(parts[0])] = {int(p) for p in parts}

# YOUR CODE HERE
# Build cone_prefix_count as a dict mapping each root_asn to the sum of
# asn_prefix_count.get(m, 0) for every m in cone_members.
cone_prefix_count = {}

cone_counts = [v for v in cone_prefix_count.values() if v > 0]
cone_avg = sum(cone_counts) / len(cone_counts) if cone_counts else 0.0
cone_median = statistics.median(cone_counts) if cone_counts else 0

print(f"cone (number of prefixes)")
print(f"   minimum:         {min(cone_counts) if cone_counts else 0}")
print(f"   maximum:         {max(cone_counts) if cone_counts else 0}")
print(f"   average per ASN: {cone_avg:.1f}")
print(f"   median per ASN:  {int(cone_median)}")

# YOUR CODE HERE
# Build cone_address_count. For each root_asn:
#   Collect all nets from asn_to_nets for every member of its cone,
#   sort by (network_address, prefixlen).
#   Walk the sorted list and sum only non-overlapping address ranges using
#   a running covered_end pointer: add net.num_addresses when
#   net_start > covered_end, then update covered_end.
cone_address_count = {}

In [ ]:
# YOUR CODE HERE
# Compute CCDF arrays for cone prefix counts.
# Using cone_counts (list of per-root-AS cone prefix counts, built above):
#   cone_prefix_dist: Counter of cone_counts
#   x_cone: sorted list of unique cone prefix count values
#   y_cone: list where y_cone[i] = number of ASes with at least x_cone[i] prefixes in their cone.
#     Hint: start remaining = len(cone_counts), subtract cone_prefix_dist[xi] at each step.
pass

# YOUR CODE HERE
# Compute CCDF arrays for cone address counts.
# Using cone_address_count (root ASN -> address count, built above):
#   cone_addr_list: address counts from cone_address_count.values(), excluding zeros
#   cone_addr_dist: Counter of cone_addr_list
#   x_addr: sorted list of unique address count values
#   y_addr: list where y_addr[i] = number of ASes with at least x_addr[i] addresses in their cone
pass

fig, ax1 = plt.subplots()
ax1.plot(x_cone, y_cone, marker=".", markersize=5, color="C0", label="by prefix count")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Number of IPv4 prefixes in customer cone")
ax1.set_ylabel("Number of ASNs")

ax2 = ax1.twiny()
ax2.plot(x_addr, y_addr, marker=".", markersize=5, color="C1", label="by address count")
ax2.set_xscale("log")
ax2.set_xlabel("Number of IPv4 addresses in customer cone")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.set_title(f"CCDF: customer cone IPv4 prefix and address count per AS ({collector})")
ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

### Question 4

What does the shape of the customer cone CCDF reveal about how IPv4 reachability is distributed across ASes?

YOUR ANSWER HERE

### Question 5

Why do the prefix-count and address-count curves diverge on the cone plot? What does this tell you about how large transit providers aggregate address space compared to their prefix count?

YOUR ANSWER HERE

### Question 6

How does the customer cone CCDF compare to the origin AS CCDF from Task 1? Why might the customer cone CCDFs converge long before the origin CCDFs?

YOUR ANSWER HERE